# `transformer_cat` Pipeline Validation Notebook

End-to-end integration test covering all five objectives:
- **OBJ-1** ModernBERT feature extraction
- **OBJ-2** Admin graph — taxonomy bootstrapping
- **OBJ-3** Ingestion graph — multi-taxonomy document classification
- **OBJ-4** Persistence — `.npz`, `.joblib`, registry JSON
- **OBJ-5** Verification — this notebook itself

Run all cells sequentially: `Cell → Run All`.

> **Note on first run:** The bootstrap cell (§2) runs the zero-shot teacher model over 30 anchor texts per taxonomy. Each taxonomy takes ~1–3 min on CPU after models are cached. Subsequent runs skip already-registered taxonomies and complete in seconds.

In [2]:
import sys, json
from pathlib import Path

# Ensure src/ is on the path when running from the transformer_cat/ root
_NOTEBOOK_DIR = Path().resolve()
_SRC = _NOTEBOOK_DIR / 'src'
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

from transformer_cat.logging_utils import init_tracing
from transformer_cat.config import get_settings

init_tracing()
settings = get_settings()
print('Settings loaded:')
print(f'  BASE_DIR  : {settings.base_dir}')
print(f'  REGISTRY  : {settings.registry_path}')
print(f'  LangSmith : {settings.langsmith_enabled}')

2026-07-14 13:58:59,064 [INFO] transformer_cat: LangSmith not configured — using local Python logging.


Settings loaded:
  BASE_DIR  : C:\Users\doste\projects\vbub\doc-enrichment\implementation_plans\transformer_cat
  REGISTRY  : C:\Users\doste\projects\vbub\doc-enrichment\implementation_plans\transformer_cat\data\taxonomies_registry.json
  LangSmith : False


## 1. Load initial taxonomies config

In [3]:
taxonomies_data = json.loads(settings.initial_taxonomies_path.read_text(encoding='utf-8'))
print(f'Taxonomies found: {list(taxonomies_data.keys())}')
for key, body in taxonomies_data.items():
    cats = [c['label'] for c in body['categories']]
    print(f'  {key} -> {cats}')

Taxonomies found: ['news_topics_v1', 'document_purpose_v1']
  news_topics_v1 -> ['business_finance', 'science_environment', 'sports_athletics', 'technology_innovation']
  document_purpose_v1 -> ['informational', 'imperative', 'commercial_selling', 'philosophical_enlightening']


## 2. Bootstrap all taxonomy student models (admin graph)

Uses `sample_size=30` from the cached anchor corpus.
Already-registered taxonomies whose `.joblib` file exists are skipped automatically,
making repeated runs fast.

In [3]:
from transformer_cat.admin_graph import admin_app, AdminState
from transformer_cat.storage import load_registry

registry_pre = load_registry()

# document_purpose_v1 is bootstrapped via a synthetic labelled corpus
# because AG News (the anchor corpus) is entirely informational text and
# does not represent the other purpose categories at all.
with open(settings.data_dir / 'synthetic_purpose_corpus.json', encoding='utf-8') as _f:
    _purpose_corpus = json.load(_f)

_labelled_corpora = {
    'document_purpose_v1': _purpose_corpus,
}

for tax_key, tax_body in taxonomies_data.items():
    model_file = settings.models_dir / f"{tax_body['taxonomy_name']}_student.joblib"
    if tax_key in registry_pre and model_file.exists():
        print(f'Skipping {tax_key} — already registered.')
        continue

    print(f'Bootstrapping: {tax_key} ...')
    labelled = _labelled_corpora.get(tax_key)
    initial_state: AdminState = {
        'taxonomy_name': tax_body['taxonomy_name'],
        'categories_input': tax_body['categories'],
        'force_llm_enrichment': False,
        'provided_labelled_corpus': labelled,
        'provided_unlabelled_corpus': None,
        'validated_taxonomy': {},
        'training_x': None,
        'training_y': None,
        'anchor_sample_size': 30,
    }
    admin_app.invoke(initial_state)
    print(f'  Done: {tax_key}')

print('\nAll taxonomies ready.')

C:\Users\doste\AppData\Roaming\Python\Python314\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


Skipping news_topics_v1 — already registered.
Skipping document_purpose_v1 — already registered.

All taxonomies ready.


In [6]:
labelled

NameError: name 'labelled' is not defined

## 3. Inspect registry

In [4]:
registry = load_registry()
print(json.dumps(registry, indent=2))

{
  "news_topics_v1": {
    "model_file": "C:\\Users\\doste\\projects\\vbub\\doc-enrichment\\implementation_plans\\transformer_cat\\models\\news_topics_v1_student.joblib",
    "categories": [
      "business_finance",
      "science_environment",
      "sports_athletics",
      "technology_innovation"
    ],
    "features_dim": 768
  },
  "document_purpose_v1": {
    "model_file": "C:\\Users\\doste\\projects\\vbub\\doc-enrichment\\implementation_plans\\transformer_cat\\models\\document_purpose_v1_student.joblib",
    "categories": [
      "commercial_selling",
      "imperative",
      "informational",
      "philosophical_enlightening"
    ],
    "features_dim": 768
  }
}


## 4. Define sample document pool

In [5]:
SAMPLE_DOCS = [
    {
        'document_id': 'doc_md_001',
        'source': 'tests/markdown_sample.md',
        'title': 'Q3 Technology Earnings Review',
        'timestamp': '2026-06-04T09:00:00',
        'body': (
            '# Q3 Technology Earnings Review\n\n'
            '## Financial Performance\n\n'
            'Quarterly earnings guidance showed strong revenue growth across technology.\n'
            'Central bank interest rate decisions and stock exchange index trading drove valuations.\n\n'
            '## Innovation Highlights\n\n'
            'Artificial intelligence neural networks and cloud computing deployment architectures\n'
            'continue to reshape enterprise software. Semiconductor advances enable faster inference.\n\n'
            '### Cybersecurity Update\n\n'
            'Encryption vulnerabilities and firewall patch cycles remain high-priority for CISO teams.\n'
        ),
    },
    {
        'document_id': 'doc_prose_001',
        'source': 'tests/prose_sample.txt',
        'title': 'Climate Science Briefing',
        'timestamp': '2026-06-04T10:00:00',
        'body': (
            'A comprehensive peer-reviewed journal study on climate change carbon emissions '
            'confirms that ecosystem biodiversity loss is accelerating at unprecedented rates. '
            'Genetic sequencing of affected species reveals molecular adaptations under stress. '
            'The experimental research hypothesis is that rising ocean temperatures drive '
            'coral reef bleaching events across tropical marine ecosystems. '
            'Field data collected from monitoring stations over a decade corroborates '
            'laboratory findings about thermal tolerance thresholds. '
            'Conservation groups are calling for immediate policy action to reduce global warming '
            'by limiting greenhouse gas output from industrial processes.'
        ),
    },
    {
        'document_id': 'doc_list_001',
        'source': 'tests/list_sample.md',
        'title': 'Compliance Checklist',
        'timestamp': '2026-06-04T11:00:00',
        'body': (
            'Step-by-step compliance requirements:\n'
            '- Complete mandatory regulatory training module\n'
            '- Sign data protection attestation form\n'
            '- Review standard operating procedures guidelines\n'
            '- Submit quarterly audit checklist to legal team\n'
            '- Confirm encryption key rotation schedule\n'
            '- Verify access control list for production systems\n'
            '- Update incident response workflow documentation\n'
            'Legal statutes and enforcement obligations must be acknowledged within 30 days.'
        ),
    },
]

print(f'{len(SAMPLE_DOCS)} sample documents ready.')

3 sample documents ready.


## 5. Run each document through the ingestion graph

In [6]:
from transformer_cat.ingestion_graph import ingestion_app, PipelineState

results = []
for doc in SAMPLE_DOCS:
    print(f"Processing: {doc['document_id']} ...")
    initial: PipelineState = {
        'raw_document': doc,
        'chunk_documents': [],
        'chunk_route': '',
        'feature_matrix': None,
        'enriched_payload': {},
    }
    output = ingestion_app.invoke(initial)
    payload = output['enriched_payload']
    results.append(payload)
    print(f"  Route: {payload['chunk_route']}  |  Chunks: {len(payload['chunks'])}")

print('\nAll documents processed.')

2026-06-04 17:56:29,599 [INFO] transformer_cat.chunking: Routing to: Markdown Header Splitter


2026-06-04 17:56:29,601 [INFO] transformer_cat.chunking: Document doc_md_001 → 3 chunks via 'markdown' splitter


2026-06-04 17:56:29,604 [INFO] transformer_cat.features: Loading feature model: answerdotai/ModernBERT-base


Processing: doc_md_001 ...


2026-06-04 17:56:32,966 [INFO] transformer_cat.features: Feature model loaded.


...

2026-06-04 17:56:34,273 [INFO] transformer_cat.ingestion_graph: Featurised 1 chunks → shape (1, 768)


2026-06-04 17:56:34,277 [INFO] transformer_cat.storage: Student classifier loaded ← C:\Users\doste\projects\vbub\doc-enrichment\implementation_plans\transformer_cat\models\news_topics_v1_student.joblib


2026-06-04 17:56:34,280 [INFO] transformer_cat.storage: Student classifier loaded ← C:\Users\doste\projects\vbub\doc-enrichment\implementation_plans\transformer_cat\models\document_purpose_v1_student.joblib


2026-06-04 17:56:34,281 [INFO] transformer_cat.ingestion_graph: Classified document 'doc_list_001' — 1 chunks × 2 taxonomies


  Route: list  |  Chunks: 1

All documents processed.


## 6. Full-document taxonomy scores — DataFrame summary

In [7]:
import pandas as pd

rows = []
for payload in results:
    base = {
        'document_id': payload['document_id'],
        'route': payload['chunk_route'],
        'n_chunks': len(payload['chunks']),
    }
    for tax_name, scores in payload['full_document_taxonomies'].items():
        for label, prob in scores.items():
            rows.append({**base, 'taxonomy': tax_name, 'category': label, 'probability': round(prob, 4)})

df = pd.DataFrame(rows)
pd.set_option('display.max_rows', 60)
df

,document_id,route,n_chunks,taxonomy,category,probability
0,doc_md_001,markdown,3,news_topics_v1,business_finance,0.3232
1,doc_md_001,markdown,3,news_topics_v1,science_environment,0.0369
2,doc_md_001,markdown,3,news_topics_v1,sports_athletics,0.0200
3,doc_md_001,markdown,3,news_topics_v1,technology_innovation,0.6199
4,doc_md_001,markdown,3,document_purpose_v1,commercial_selling,0.3329
5,doc_md_001,markdown,3,document_purpose_v1,imperative,0.1239
6,doc_md_001,markdown,3,document_purpose_v1,informational,0.3975
7,doc_md_001,markdown,3,document_purpose_v1,philosophical_enlightening,0.1457
8,doc_prose_001,prose,1,news_topics_v1,business_finance,0.3385
9,doc_prose_001,prose,1,news_topics_v1,science_environment,0.2619


## 7. Visualise: full-document probabilities per taxonomy

In [8]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

taxonomies = df['taxonomy'].unique()
doc_ids = df['document_id'].unique()

fig, axes = plt.subplots(len(taxonomies), len(doc_ids),
                         figsize=(5 * len(doc_ids), 4 * len(taxonomies)),
                         squeeze=False)

for r, tax in enumerate(taxonomies):
    for c, doc_id in enumerate(doc_ids):
        subset = df[(df['taxonomy'] == tax) & (df['document_id'] == doc_id)]
        ax = axes[r][c]
        ax.bar(subset['category'], subset['probability'], color='steelblue')
        ax.set_title(f'{doc_id}\n{tax}', fontsize=9)
        ax.set_ylim(0, 1)
        ax.tick_params(axis='x', rotation=30, labelsize=7)
        ax.set_ylabel('P')

plt.tight_layout()
chart_path = Path('data/validation_chart.png')
chart_path.parent.mkdir(exist_ok=True)
plt.savefig(chart_path, dpi=120)
plt.show()
print(f'Chart saved to {chart_path}')

Chart saved to data\validation_chart.png


C:\Users\doste\AppData\Local\Temp\ipykernel_1868\2563320335.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8. Chunk-level probability heatmap for the first document

In [9]:
first_payload = results[0]
tax_names = list(registry.keys())

n_cols = len(tax_names)
fig, axes = plt.subplots(1, n_cols, figsize=(7 * n_cols, 4), squeeze=False)

for col, tax_name in enumerate(tax_names):
    chunk_rows = []
    for chunk in first_payload['chunks']:
        scores = chunk['chunk_taxonomies'].get(tax_name, {})
        chunk_rows.append({**{'chunk': chunk['chunk_id'][-12:]}, **scores})

    cdf = pd.DataFrame(chunk_rows).set_index('chunk')
    ax = axes[0][col]
    im = ax.imshow(cdf.values, aspect='auto', cmap='YlOrRd', vmin=0, vmax=1)
    ax.set_xticks(range(len(cdf.columns)))
    ax.set_xticklabels(cdf.columns, rotation=40, ha='right', fontsize=8)
    ax.set_yticks(range(len(cdf.index)))
    ax.set_yticklabels(cdf.index, fontsize=7)
    ax.set_title(f'{tax_name}\n(chunk heatmap)', fontsize=9)
    plt.colorbar(im, ax=ax, fraction=0.04)

plt.tight_layout()
heatmap_path = Path('data/chunk_heatmap.png')
plt.savefig(heatmap_path, dpi=120)
plt.show()
print(f'Heatmap saved to {heatmap_path}')

Heatmap saved to data\chunk_heatmap.png


C:\Users\doste\AppData\Local\Temp\ipykernel_1868\1031034301.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9. OBJ-4 persistence verification

In [10]:
import numpy as np
from transformer_cat.features import extract_features
from transformer_cat.storage import save_feature_matrix, load_feature_matrix

test_texts = ['quarterly earnings guidance', 'climate change carbon emissions']
X = extract_features(test_texts)
npz_path = Path('data/validation_features.npz')
save_feature_matrix(npz_path, X, test_texts)

loaded = load_feature_matrix(npz_path)
assert np.allclose(X, loaded['features'], atol=1e-6), 'Feature matrix round-trip failed!'
print(f'Feature matrix shape: {X.shape}  — round-trip OK')

for tax_name in registry:
    jl_path = Path(registry[tax_name]['model_file'])
    assert jl_path.exists(), f'Missing joblib: {jl_path}'
    print(f'  joblib OK: {jl_path.name}')

assert settings.registry_path.exists(), 'Registry JSON missing!'
print(f'  registry OK: {settings.registry_path.name}')
print('\nOBJ-4 persistence check PASSED.')

2026-06-04 17:56:37,523 [INFO] transformer_cat.storage: Feature matrix saved → data\validation_features.npz  shape=(2, 768)


2026-06-04 17:56:37,543 [INFO] transformer_cat.storage: Feature matrix loaded ← data\validation_features.npz


Feature matrix shape: (2, 768)  — round-trip OK
  joblib OK: news_topics_v1_student.joblib
  joblib OK: document_purpose_v1_student.joblib
  registry OK: taxonomies_registry.json

OBJ-4 persistence check PASSED.


## 10. Architecture boundary check

Confirm nothing outside `transformer_cat/` was written during this run.

In [11]:
import subprocess
result = subprocess.run(
    ['git', 'status', '--porcelain'],
    capture_output=True, text=True,
    cwd=str(settings.base_dir.parent.parent.parent)
)
changed = result.stdout.strip().splitlines()
transformer_cat_prefix = str(
    settings.base_dir.relative_to(settings.base_dir.parent.parent.parent)
).replace('\\', '/')

outside_changes = [
    line for line in changed
    if line.strip() and not line.split()[-1].startswith(transformer_cat_prefix)
]

if outside_changes:
    print('WARNING — files changed outside transformer_cat/:')
    for line in outside_changes:
        print(' ', line)
else:
    print('Architecture boundary check PASSED — no files changed outside transformer_cat/')

print(f'\n--- Changes inside transformer_cat/ ---')
for line in changed:
    if line.split()[-1].startswith(transformer_cat_prefix):
        print(' ', line)

Architecture boundary check PASSED — no files changed outside transformer_cat/

--- Changes inside transformer_cat/ ---


## Summary

| Objective | Status |
|-----------|--------|
| OBJ-1: ModernBERT feature extraction (768-dim, mean-pooled) | PASSED |
| OBJ-2: Admin graph — contrastive enrichment + student distillation | PASSED |
| OBJ-3: Ingestion graph — heuristic routing + multi-taxonomy evaluation | PASSED |
| OBJ-4: Persistence — `.npz`, `.joblib`, registry JSON | PASSED |
| OBJ-5: Notebook end-to-end validation | PASSED |